### 원하는 비율 조합 파일 생성 
- total reads: 5000 reads
- GBM percent: 0.1%, 0.5%, 1%, 2%, 2.5%, 3%, 5%, 10%, 100%

In [ ]:
### bed파일 생성 ###

import os
import subprocess

mut_ratios = [0, 5, 25, 50, 100, 125, 150, 250, 500, 5000]  # List of mutation ratios
OUTPUT_BASE = "./Methylation/results/step04_ML classifier/"  # 출력 파일 기본 경로

def process_bam(mut_reads, i):
    input_bam = f"./Methylation/results/step03_Preprocessing for ML/mut_{mut_reads}_reads/sampled_reads_{i}.bam"
    output_dir = f"{OUTPUT_BASE}/mut_{mut_reads}_reads"
    methylation_output_dir = f"{output_dir}"    
    if not os.path.exists(input_bam):
        print(f"Error: {input_bam} does not exist.")
        return    
    # Ensure output directories exist
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(methylation_output_dir, exist_ok=True)
    methylation_output_file = f"{methylation_output_dir}/methylation_data_{i}.bedGraph.gz"
    # Bismark methylation extractor command
    bismark_cmd = [
        "bismark_methylation_extractor", 
        "--paired-end", 
        "--ignore_3prime", "1",
        "--genome_folder", "./data/Homo_sapiens.GRCh38.dna.primary_assembly.fa",
        "--bedGraph", 
        "--gzip", 
        "--output_dir", methylation_output_dir, 
        input_bam
    ]
    print("Running command:", " ".join(bismark_cmd)) 
    try:
        subprocess.run(bismark_cmd, check=True)
        print(f"Methylation data saved to {methylation_output_file}")
    except subprocess.CalledProcessError as e:
        print(f"Error during Bismark methylation extraction: {e}") 
    print(f"Completed mut_reads={mut_reads}, i={i}")

# Main processing loop for each mutation ratio and file index
for mut_reads in mut_ratios:
    for i in range(1, 1001):  
        process_bam(mut_reads, i)


